# House Price Estimation Using Regression - Exploratory Data Analysis

**Course / Capstone**: Applied Machine Learning & Predictive Modeling  
**Dataset**: Ames Housing Dataset (Dean De Cock, 2011 / OpenML ID: 43926)  
**Target Variable**: Historical Sale Price (`SalePrice` in USD)  
**Geographic Scope**: Ames, Iowa, United States (2006–2010 transactions)  

---

## 1. Environment Setup & Data Loading
We import our core data science libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and configure publication-quality visualization styles.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Palette and style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
PALETTE = {
    'teal': '#247F7B',
    'charcoal': '#293332',
    'amber': '#D99A3D',
    'bg': '#F7F5F0'
}

DATA_PATH = Path('../data/ames_housing_clean.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/ames_housing_clean.csv')

df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 2. Dataset Overview & Descriptive Statistics
Let us inspect column datatypes, non-null counts, and summary statistics across all numeric features.

In [ ]:
print("--- Data Types and Missing Values ---")
print(df.info())

print("\n--- Descriptive Statistics ---")
df.describe().round(2)

## 3. Target Distribution: Historical Sale Price (`SalePrice`)
Examining the empirical distribution of property prices in USD. Residential real estate values typically exhibit positive (right-hand) skewness.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['SalePrice'], kde=True, color=PALETTE['teal'], bins=40, ax=ax)

mean_val = df['SalePrice'].mean()
median_val = df['SalePrice'].median()

ax.axvline(mean_val, color=PALETTE['amber'], linestyle='--', linewidth=2, label=f'Mean: ${mean_val:,.0f}')
ax.axvline(median_val, color=PALETTE['charcoal'], linestyle='-', linewidth=2, label=f'Median: ${median_val:,.0f}')

ax.set_title('Distribution of Residential Property Sale Prices (Ames, Iowa)', fontsize=14, weight='bold', pad=12)
ax.set_xlabel('Sale Price in USD ($)', fontsize=11)
ax.set_ylabel('Property Count', fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

### Observation:
The median house price is approximately $160,000 USD, while the mean is slightly higher ($180,000 USD) due to luxury home sales exceeding $500,000 USD. This demonstrates classic right-skew.

## 4. Property Footprint Analysis: Living Area & Lot Area vs. Price
How does usable indoor space (`GrLivArea`) and parcel area (`LotArea`) relate to final transaction price?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. GrLivArea vs Price
sns.scatterplot(
    data=df,
    x='GrLivArea',
    y='SalePrice',
    hue='OverallQual',
    palette='viridis',
    alpha=0.7,
    ax=ax1
)
ax1.set_title('Above-Ground Living Area vs. Sale Price', fontsize=13, weight='bold')
ax1.set_xlabel('Living Area (Sq. Ft.)', fontsize=11)
ax1.set_ylabel('Sale Price (USD)', fontsize=11)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

# 2. LotArea vs Price (Log-scale x for lot area due to agricultural/acreage parcels)
sns.scatterplot(
    data=df,
    x='LotArea',
    y='SalePrice',
    color=PALETTE['teal'],
    alpha=0.6,
    ax=ax2
)
ax2.set_xscale('log')
ax2.set_title('Lot Area vs. Sale Price (Logarithmic Scale)', fontsize=13, weight='bold')
ax2.set_xlabel('Lot Area (Sq. Ft., Log Scale)', fontsize=11)
ax2.set_ylabel('Sale Price (USD)', fontsize=11)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

plt.tight_layout()
plt.show()

## 5. Structural Vintage: House Age at Sale vs. Price
We computed `HouseAge` as `YrSold - YearBuilt`. How does the age of the structure impact market valuation?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(
    data=df,
    x='HouseAge',
    y='SalePrice',
    color=PALETTE['amber'],
    alpha=0.6,
    ax=ax
)
sns.regplot(
    data=df,
    x='HouseAge',
    y='SalePrice',
    scatter=False,
    color=PALETTE['charcoal'],
    ax=ax
)
ax.set_title('House Age at Time of Sale vs. Transaction Price', fontsize=13, weight='bold', pad=12)
ax.set_xlabel('House Age at Sale (Years: YrSold - YearBuilt)', fontsize=11)
ax.set_ylabel('Sale Price (USD)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
plt.tight_layout()
plt.show()

## 6. Neighborhood Stratification Analysis
Ames neighborhoods differ substantially by school district, proximity to Iowa State University, and era of development.

In [ ]:
plt.figure(figsize=(14, 6))
order = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index
sns.boxplot(
    data=df,
    x='Neighborhood',
    y='SalePrice',
    order=order,
    palette='crest'
)
plt.title('Sale Price Distribution Across Ames Neighborhoods (Sorted by Median Price)', fontsize=14, weight='bold', pad=12)
plt.xlabel('Neighborhood Code', fontsize=11)
plt.ylabel('Sale Price (USD)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
plt.tight_layout()
plt.show()

## 7. Overall Quality, Rooms, and Garage Capacity
Analyzing discrete structural and amenity features.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. OverallQual
sns.boxplot(data=df, x='OverallQual', y='SalePrice', ax=axes[0, 0], color=PALETTE['teal'])
axes[0, 0].set_title('Sale Price by Overall Quality (1-10)', weight='bold')
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

# 2. FullBath
sns.boxplot(data=df, x='FullBath', y='SalePrice', ax=axes[0, 1], color=PALETTE['amber'])
axes[0, 1].set_title('Sale Price by Full Bathrooms Above Ground', weight='bold')
axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

# 3. BedroomAbvGr
sns.boxplot(data=df, x='BedroomAbvGr', y='SalePrice', ax=axes[1, 0], color=PALETTE['charcoal'])
axes[1, 0].set_title('Sale Price by Bedrooms Above Ground', weight='bold')
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

# 4. GarageCars
sns.boxplot(data=df, x='GarageCars', y='SalePrice', ax=axes[1, 1], color=PALETTE['teal'])
axes[1, 1].set_title('Sale Price by Garage Capacity (Cars)', weight='bold')
axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))

plt.tight_layout()
plt.show()

## 8. Correlation Heatmap
Let us quantify the pairwise linear correlation between numeric features and the target variable.

In [ ]:
num_cols = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'FullBath', 'HouseAge', 'LotArea', 'BedroomAbvGr']
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Pearson Correlation Heatmap (Selected Features)', fontsize=13, weight='bold', pad=12)
plt.tight_layout()
plt.show()

## 9. Key Findings & Methodological Summary

### Primary Insights:
1. **Dominant Predictor**: `OverallQual` has the strongest correlation ($r \approx 0.80$) with `SalePrice`. The step-up in median price between quality level 6, 7, 8, and 9 is dramatic.
2. **Living Area**: `GrLivArea` is the second strongest driver ($r \approx 0.71$). Indoor finished space consistently predicts higher transaction values.
3. **Garage & Bathrooms**: Both `GarageCars` ($r \approx 0.64$) and `FullBath` ($r \approx 0.54$) provide strong additive signals.
4. **Vintage Effect**: `HouseAge` exhibits negative correlation ($r \approx -0.56$), confirming newer builds trade at marked premiums.

> **Crucial Methodological Caveat**: Correlation does **not** prove causation. For instance, having 4 full bathrooms does not inherently create $100,000 of value on its own; rather, large bathrooms are typically constructed within high-end, high-square-footage homes situated in affluent neighborhoods.